In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import LabelBinarizer
from tqdm.auto import tqdm

/Users/kk985/PhD/Research/LessDiscriminatoryAllocations/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
N_SEEDS           = 5
N_MODELS          = 1000
TEST_SIZE         = 0.20
VALIDATION_SIZE   = 0.25
BOOTSTRAP_SIZE    = 0.50
NEED_COL          = "gagne_sum_t"
MODEL_CLASS       = "nn_multiclass"

# variables
METHOD            = "bootstrap"
NEED_THRESHOLD    = 1
QUALIFICATION_COL = f"qualified_gagne_{NEED_THRESHOLD}"
PRED_METHOD       = "average"

ROOT         = Path.cwd().parent
fp_input     = ROOT / "data/obermeyer/obermeyer_data_cleaned.csv"
fp_output    = ROOT / f"data/predictions/test_{METHOD}_{MODEL_CLASS}_{PRED_METHOD}_{QUALIFICATION_COL}.csv"
fp_nn_losses = ROOT / f"data/predictions/losses/test_{METHOD}_{MODEL_CLASS}_{PRED_METHOD}_{QUALIFICATION_COL}.csv"

# Optional full distributions are written one model at a time to avoid holding
# 1,000 distributions in memory. The main CSV always contains probabilities and expected need.
SAVE_DISTRIBUTIONS = False
fp_distributions = ROOT / f"data/predictions/test_{METHOD}_{MODEL_CLASS}_{PRED_METHOD}_{QUALIFICATION_COL}_distributions"

In [3]:
df = pd.read_csv(fp_input)
NEED_BOUND = df[NEED_COL].max()

In [4]:
# SETUP VARIABLES: same feature order and metadata as the bootstrap notebook.
features = [
    'dem_female', 'dem_age_band_18-24_tm1', 'dem_age_band_25-34_tm1',
    'dem_age_band_35-44_tm1', 'dem_age_band_45-54_tm1',
    'dem_age_band_55-64_tm1', 'dem_age_band_65-74_tm1', 'dem_age_band_75+_tm1',
    'hypertension_elixhauser_tm1', 'cost_dialysis_tm1', 'cost_emergency_tm1',
    'cost_home_health_tm1', 'cost_ip_medical_tm1', 'cost_ip_surgical_tm1',
    'cost_laboratory_tm1', 'cost_op_primary_care_tm1', 'cost_op_specialists_tm1',
    'cost_op_surgery_tm1', 'cost_other_tm1', 'cost_pharmacy_tm1',
    'cost_physical_therapy_tm1', 'cost_radiology_tm1', 'gagne_sum_tm1',
]
other_variables = ['person_id', 'gagne_sum_t', 'cost_t']

In [5]:
def validate_needs(values, bound):
    values = np.asarray(values, dtype=float)
    if (not np.isfinite(values).all() or (values < 0).any() or (values > bound).any() or (values != np.floor(values)).any()):
        raise ValueError(f'Need labels must be finite integers in 0 - {bound}.')
    return values.astype(np.int64)


X = df[features + other_variables]
h = pd.Series(validate_needs(df[NEED_COL], NEED_BOUND), index=df.index, name=NEED_COL)
y = df[QUALIFICATION_COL]

NEED_CLASSES = np.arange(NEED_BOUND + 1)

In [6]:
class MulticlassNeedMLP(MLPClassifier):
    def __init__(self, need_bound, random_state=None, max_iter=1000):
        self.need_bound = need_bound
        super().__init__(hidden_layer_sizes=(50, 100, 200), random_state=random_state, max_iter=max_iter)

    def _validate_input(self, X, y, incremental, reset):
        # Retain sklearn's input validation; replace only the target encoding.
        X, _ = super()._validate_input(X, y, incremental, reset)
        labels = validate_needs(y, self.need_bound)
        self._label_binarizer = LabelBinarizer().fit(np.arange(self.need_bound + 1))
        self.classes_ = self._label_binarizer.classes_
        return X, self._label_binarizer.transform(labels).astype(bool)


def get_bootstrap_model(X_train, h_train, seed, bootstrap_size=BOOTSTRAP_SIZE):
    np.random.seed(seed)
    idx = np.random.permutation(len(X_train))[:int(len(X_train) * bootstrap_size)]
    if len(idx) == 0:
        raise ValueError('The sampled training subset is empty.')
    model = MulticlassNeedMLP(need_bound=NEED_BOUND, random_state=seed, max_iter=1000)
    model.fit(X_train[idx], h_train[idx])
    return model


def derive_predictions(model, X_values, method="average"):
    if not np.array_equal(model.classes_, NEED_CLASSES):
        raise ValueError('Unexpected class order or incomplete output support.')
    q = model.predict_proba(X_values)
    p = np.clip(q[:, NEED_CLASSES >= NEED_THRESHOLD].sum(axis=1), 0.0, 1.0)
    if method == "average":
        mu = q @ NEED_CLASSES.astype(float)
    elif method == "argmax":
        mu = q.argmax(axis=1)
    else:
        raise ValueError(f"method needs to be either 'average' or 'argmax'. Provided method: {method}")
    return q, p, mu

In [ ]:
output = []
nn_losses = []
if SAVE_DISTRIBUTIONS:
    fp_distributions.mkdir(parents=True, exist_ok=True)

for seed in range(N_SEEDS):
    X_train, X_test, y_train, y_test, h_train, h_test = train_test_split(X, y, h, test_size=TEST_SIZE, random_state=seed)
    X_train, X_val, y_train, y_val, h_train, h_val    = train_test_split(X_train, y_train, h_train, test_size=VALIDATION_SIZE, random_state=seed)

    cost      = X_test['cost_t'].to_numpy()
    gagne     = X_test['gagne_sum_t'].to_numpy()
    person_id = X_test['person_id'].to_numpy()

    X_train = X_train.drop(columns=other_variables).to_numpy().copy()
    X_val   = X_val.drop(columns=other_variables).to_numpy().copy()
    X_test  = X_test.drop(columns=other_variables).to_numpy().copy()
    h_train = h_train.to_numpy().copy()
    h_val   = h_val.to_numpy().copy()
    h_test  = h_test.to_numpy().copy()
    y_train = y_train.to_numpy().copy()
    y_val   = y_val.to_numpy().copy()
    y_test  = y_test.to_numpy().copy()

    predictions, expected_needs = {}, {}
    training_loss, validation_loss = {}, {}
    for i in tqdm(range(N_MODELS), desc=f"seed {seed}"):
        model = get_bootstrap_model(X_train, h_train, seed=i, bootstrap_size=BOOTSTRAP_SIZE)
        
        q_test,  p_test,  mu_test = derive_predictions(model, X_test)
        q_train, p_train, _       = derive_predictions(model, X_train)
        q_val,   p_val,   _       = derive_predictions(model, X_val)

        model_name = f'm_{i + 1}'
        predictions[model_name] = p_test
        expected_needs[f'exp_need_{model_name}'] = mu_test
        training_loss[model_name] = log_loss(y_train, p_train, labels=[0, 1])
        validation_loss[model_name] = log_loss(y_val, p_val, labels=[0, 1])

        training_cross_entropy_loss = log_loss(h_train, q_train, labels=NEED_CLASSES)
        validation_cross_entropy_loss = log_loss(h_val, q_val, labels=NEED_CLASSES)
        test_cross_entropy_loss = log_loss(h_test, q_test, labels=NEED_CLASSES)
        nn_losses.append({
            'seed': seed, 'model': model_name,
            'training_cross_entropy': training_cross_entropy_loss,
            'validation_cross_entropy': validation_cross_entropy_loss,
            'test_cross_entropy': test_cross_entropy_loss,
            'training_log_loss': training_cross_entropy_loss,
            'validation_log_loss': validation_cross_entropy_loss,
            'n_train_sample': int(len(X_train) * BOOTSTRAP_SIZE),
            'n_iter': model.n_iter_,
        })
        if SAVE_DISTRIBUTIONS:
            np.savez_compressed(
                fp_distributions / f'seed-{seed}_{model_name}.npz',
                person_id=person_id, classes=NEED_CLASSES, q=q_test)

    df_pred = pd.concat([
        pd.DataFrame(predictions),
        pd.DataFrame(training_loss, index=[0]),
        pd.DataFrame(validation_loss, index=[0]),
    ]).reset_index(drop=True)
    df_pred['y'] = np.concatenate([y_test, [np.nan, np.nan]])
    df_pred['person_id'] = np.concatenate([person_id, [-2, -1]])
    df_pred['cost_t'] = np.concatenate([cost, [np.nan, np.nan]])
    df_pred['gagne_sum_t'] = np.concatenate([gagne, [np.nan, np.nan]])
    df_pred['seed'] = seed

    # Add one matched mean per model; the two loss rows have no expected need.
    df_need = pd.DataFrame(expected_needs).reindex(df_pred.index)
    df_pred = pd.concat([df_pred, df_need], axis=1)
    output.append(df_pred)

df_output = pd.concat(output)
for c in df_output.columns:
    if c.startswith('m_'):
        df_output[c] = df_output[c].astype('float32')
df_nn_losses = pd.DataFrame(nn_losses)

seed 4: 100%|██████████| 100/100 [43:52<00:00, 26.32s/it]


In [10]:
# Inspect q-based cross entropy for every split seed and model.
df_nn_losses[['seed', 'model', 'training_cross_entropy', 'validation_cross_entropy', 'test_cross_entropy']]

,seed,model,training_cross_entropy,validation_cross_entropy,test_cross_entropy
0,0,m_1,1.009996,1.414117,1.432650
1,0,m_2,0.947466,1.258107,1.222897
2,0,m_3,1.050142,1.498190,1.510829
3,0,m_4,0.973566,1.366496,1.350558
4,0,m_5,1.092948,1.554452,1.603251
...,...,...,...,...,...
495,4,m_96,1.009535,1.396679,1.331705
496,4,m_97,0.939851,1.161162,1.107442
497,4,m_98,0.998352,1.325708,1.280011
498,4,m_99,0.982408,1.317202,1.251261


In [9]:
fp_output.parent.mkdir(parents=True, exist_ok=True)
df_output.to_csv(fp_output, index=False)
fp_nn_losses.parent.mkdir(parents=True, exist_ok=True)
df_nn_losses.to_csv(fp_nn_losses, index=False)
print('Saved predictions:', fp_output.resolve())
print('Saved NN losses:', fp_nn_losses.resolve())

Saved predictions: /Users/kk985/PhD/Research/LessDiscriminatoryAllocations/data/predictions/test_bootstrap_nn_multiclass_large_average_qualified_gagne_1.csv
Saved NN losses: /Users/kk985/PhD/Research/LessDiscriminatoryAllocations/data/predictions/losses/test_bootstrap_nn_multiclass_large_average_qualified_gagne_1.csv
